In [ ]:
!pip install -q matplotlib seaborn scikit-learn

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

DATASET_DIR = "/content/drive/MyDrive/Colab Notebooks/food_dataset_complete_two"

TRAIN_DIR = os.path.join(DATASET_DIR, 'train')
VALIDATE_DIR = os.path.join(DATASET_DIR, 'val')
TEST_DIR = os.path.join(DATASET_DIR, 'test')

print("=======Dataset Summary=======")

for split in ['train', 'val', 'test']:
    print("\n", split)

    path = os.path.join(DATASET_DIR, split)

    for cls in os.listdir(path):
        cls_path = os.path.join(path, cls)


        if not os.path.isdir(cls_path):
            continue

        count = len(os.listdir(cls_path))
        print(cls, ":", count, "photo")

=======Dataset Summary=======

 train
larbchicken : 103 photo
larbbeef : 99 photo
massamancurrybeef : 77 photo
pad kra pao beef : 80 photo
chicken green curry : 100 photo
tom yum kung nam sai : 81 photo
pad kra pao mince pork : 120 photo
panangbeef : 87 photo
tom yum kung nam kon : 100 photo
pad thai pork : 80 photo
pad kra pao crispy pork : 80 photo
beef green curry : 80 photo
tom yum seafood : 92 photo
panangpork : 112 photo
larbpork : 118 photo
tomkhagai : 117 photo
pad thai chicken : 88 photo
gaengsomkung : 88 photo
massamancurrychicken : 93 photo
gaengsomfish : 88 photo
som tum thai : 80 photo
gaeng_som_cha_om_kung : 120 photo
som tum poo plara : 120 photo
panangchicken : 87 photo
pad thai goong : 132 photo
tom yum pla : 92 photo

 val
pad kra pao beef : 10 photo
larbbeef : 12 photo
panangpork : 14 photo
pad kra pao mince pork : 15 photo
pad kra pao crispy pork : 10 photo
tom yum kung nam kon : 13 photo
pad thai chicken : 11 photo
massamancurrybeef : 10 photo
pad thai pork : 10 ph

In [ ]:
IMG_SIZE = 224           # EfficientNetB0 ใช้ 224x224
BATCH_SIZE  = 16         # แบ่งรูปเทรนต่อรอบ
EPOCHS_TOP  = 15
EPOCHS_FINE = 15         # Fine-tune ทั้ง model
LR_TOP      = 5e-4       # Learning rate ช่วงแรก
LR_FINE     = 1e-4       # Learning rate ช่วง fine-tune
OUTPUT_DIR  = '/content/benchmark_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAME = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d)) and not d.startswith('.')])
NUM_CLASS = len(CLASS_NAME)

print("=======Config=======")
print(f' Image size  : {IMG_SIZE}x{IMG_SIZE}')
print(f' Batch size : {BATCH_SIZE}')
print(f' Epochs top : {EPOCHS_TOP}')
print(f' Epochs fine: {EPOCHS_FINE}')
print(f' Learning rate top : {LR_TOP}')
print(f' Learning rate fine: {LR_FINE}')
print(f' Amout Class  : {NUM_CLASS}')
print(f' Class name : {CLASS_NAME}')

=======Config=======
 Image size  : 224x224
 Batch size : 16
 Epochs top : 15
 Epochs fine: 15
 Learning rate top : 0.0005
 Learning rate fine: 0.0001
 Amout Class  : 26
 Class name : ['beef green curry', 'chicken green curry', 'gaeng_som_cha_om_kung', 'gaengsomfish', 'gaengsomkung', 'larbbeef', 'larbchicken', 'larbpork', 'massamancurrybeef', 'massamancurrychicken', 'pad kra pao beef', 'pad kra pao crispy pork', 'pad kra pao mince pork', 'pad thai chicken', 'pad thai goong', 'pad thai pork', 'panangbeef', 'panangchicken', 'panangpork', 'som tum poo plara', 'som tum thai', 'tom yum kung nam kon', 'tom yum kung nam sai', 'tom yum pla', 'tom yum seafood', 'tomkhagai']


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input
import os

TEST_DIR = os.path.join(DATASET_DIR, 'test')

train_data = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    rotation_range = 20,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    shear_range = 0.2,
    zoom_range = 0.2,
    horizontal_flip = True,
    brightness_range = [0.8, 1.2],
    fill_mode = 'nearest'
)

val_data = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

test_data = ImageDataGenerator(
    preprocessing_function = preprocess_input
)

# load
train_loader = train_data.flow_from_directory(
    TRAIN_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    shuffle=True
)

val_loader = val_data.flow_from_directory(
    VALIDATE_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    shuffle = False
)

test_loader = test_data.flow_from_directory(
    TEST_DIR,
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    shuffle = False
)

print('Class indices:', train_loader.class_indices)

Found 2513 images belonging to 26 classes.
Found 316 images belonging to 26 classes.
Found 322 images belonging to 26 classes.
Class indices: {'beef green curry': 0, 'chicken green curry': 1, 'gaeng_som_cha_om_kung': 2, 'gaengsomfish': 3, 'gaengsomkung': 4, 'larbbeef': 5, 'larbchicken': 6, 'larbpork': 7, 'massamancurrybeef': 8, 'massamancurrychicken': 9, 'pad kra pao beef': 10, 'pad kra pao crispy pork': 11, 'pad kra pao mince pork': 12, 'pad thai chicken': 13, 'pad thai goong': 14, 'pad thai pork': 15, 'panangbeef': 16, 'panangchicken': 17, 'panangpork': 18, 'som tum poo plara': 19, 'som tum thai': 20, 'tom yum kung nam kon': 21, 'tom yum kung nam sai': 22, 'tom yum pla': 23, 'tom yum seafood': 24, 'tomkhagai': 25}


In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications import InceptionV3

from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.resnet_v2  import preprocess_input as res_pre
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input as mob_pre
from tensorflow.keras.applications.densenet import preprocess_input as dense_pre
from tensorflow.keras.applications.inception_v3 import preprocess_input as inc_pre

MODELS_CONFIG = [
    {
        'name':         'EfficientNetB0',
        'base_class':   EfficientNetB0,
        'preprocess':   eff_pre,
        'img_size':     224,
    },

    {
        'name':         'ResNet50V2',
        'base_class':   ResNet50V2,
        'preprocess':   res_pre,
        'img_size':     224,
    },

    {
        'name':         'MobileNetV3Large',
        'base_class':   MobileNetV3Large,
        'preprocess':   mob_pre,
        'img_size':     224,
    },

    {
        'name':         'DenseNet121',
        'base_class':   DenseNet121,
        'preprocess':   dense_pre,
        'img_size':     224,
    },

    {
        'name':         'InceptionV3',
        'base_class':   InceptionV3,
        'preprocess':   inc_pre,
        'img_size':     299,
    },
]
print(f'Number of models to training: {len(MODELS_CONFIG)} ')

Number of models to training: 5 


In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def make_loaders(preprocess_function, img_size):
    train_data = ImageDataGenerator(
        preprocessing_function = preprocess_function,
        rotation_range = 20,
        width_shift_range = 0.2,
        height_shift_range = 0.2,
        shear_range = 0.2,
        zoom_range = 0.2,
        horizontal_flip = True,
        brightness_range = [0.8, 1.2],
        fill_mode = 'nearest'
    )

    val_data = ImageDataGenerator(
        preprocessing_function = preprocess_function
    )

    test_data = ImageDataGenerator(
        preprocessing_function = preprocess_function
    )

    train_loader = train_data.flow_from_directory(
        TRAIN_DIR,
        target_size=(img_size, img_size),
        batch_size = BATCH_SIZE,
        class_mode = 'categorical',
        shuffle = True
    )

    val_loader = val_data.flow_from_directory(
        VALIDATE_DIR,
        target_size=(img_size, img_size),
        batch_size = BATCH_SIZE,
        class_mode = 'categorical',
        shuffle = False
    )

    test_loader = test_data.flow_from_directory(
        TEST_DIR,
        target_size=(img_size, img_size),
        batch_size = BATCH_SIZE,
        class_mode = 'categorical',
        shuffle = False
    )
    return train_loader, val_loader, test_loader

def create_model(base_model_class, img_size, num_classes):
    base_model = base_model_class(
        weights ='imagenet',
        include_top = False,
        input_shape = (img_size, img_size, 3)
    )

    base_model.trainable = False

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation ='relu')(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(num_classes, activation ='softmax')(x)

    model = models.Model(
        inputs = base_model.input,
        outputs = output

    )

    return model, base_model


# callbacks save model
def get_callbacks(model_name):
    save_folder = "models"
    os.makedirs(save_folder, exist_ok = True)

    save_path = os.path.join(save_folder, model_name + ".keras") # Changed .h5 to .keras

    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath = save_path,
            monitor = 'val_accuracy',
            save_best_only = True
        ),
        tf.keras.callbacks.EarlyStopping(
            patience = 3,
            restore_best_weights = True
        )
    ]


# train
def train_model(config):
    model_name = config['name']
    img_size = config['img_size']

    print("Train model:", model_name)

    # โหลด data
    train_data, val_data, test_data = make_loaders(config['preprocess'], img_size)

    # สร้าง model
    model, base_model = create_model(config['base_model'], img_size, NUM_CLASS)

    # Phase 1
    print("=====Phase 1 (train head)=====")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR_TOP),
        loss = 'categorical_crossentropy',
        metrics = ['accuracy']
    )

    history1 = model.fit(
        train_data,
        validation_data = val_data,
        epochs = EPOCHS_TOP,
        callbacks = get_callbacks(model_name + "_phase1")
    )

    # Phase 2
    print("=====Phase 2 (fine-tune)=====")

    base_model.trainable = True

    model.compile(
        optimizer = tf.keras.optimizers.Adam(LR_FINE),
        loss = 'categorical_crossentropy',
        metrics = ['accuracy']
    )

    history2 = model.fit(
        train_data,
        validation_data=val_data,
        epochs=EPOCHS_FINE,
        callbacks=get_callbacks(model_name)
    )

    return model, test_data, history1, history2

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

def evaluate_model(model, test_data):
    print("=====Evaluate model=====")

    model_outputs = []
    correct_answers = []

    for i in range(len(test_data)):
        x, y = test_data[i]
        preds = model.predict(x)
        model_outputs.append(preds)
        correct_answers.append(np.argmax(y, axis=1))


    model_outputs = np.vstack(model_outputs)
    correct_answers = np.concatenate(correct_answers)


    predicted_class = np.argmax(model_outputs, axis=1)

    # Accuracy
    accuracy = np.mean(predicted_class == correct_answers) * 100

    # F1-score
    f1 = f1_score(correct_answers, predicted_class, average='macro') * 100

    # confusion_matrix
    cm = confusion_matrix(correct_answers, predicted_class)

    print("Accuracy:", round(accuracy, 2), "%")
    print("F1-score:", round(f1, 2), "%")
    print("Confusion Matrix:", cm)

    return accuracy, f1, cm

In [ ]:
import os
import gc
import pandas as pd
import tensorflow as tf

from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.resnet_v2 import preprocess_input as res_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.inception_v3 import preprocess_input as inc_pre
from tensorflow.keras.applications.densenet import preprocess_input as dense_pre

save_folder = "models"
os.makedirs(save_folder, exist_ok=True)

model_list = [
    ("EfficientNetB0", tf.keras.applications.EfficientNetB0, 224, eff_pre),
    ("MobileNetV2",    tf.keras.applications.MobileNetV2,    224, mob_pre),
    ("ResNet50",       tf.keras.applications.ResNet50,        224, res_pre),
    ("DenseNet121",    tf.keras.applications.DenseNet121,     224, dense_pre),
    ("InceptionV3",    tf.keras.applications.InceptionV3,     299, inc_pre),
]

results = []

for name, base_model, img_size, preprocess in model_list:
    config = {
        'name': name,
        'img_size': img_size,
        'base_model': base_model,
        'preprocess': preprocess,
    }

    model, test_data, h1, h2 = train_model(config)

    acc, f1, cm = evaluate_model(model, test_data)
    print(name, "→", acc, f1)

    results.append((name, acc, f1))

    model_path = os.path.join(save_folder, name + ".keras")
    model.save(model_path)
    print("Saved model →", model_path)

    # ล้าง memory หลังแต่ละ model
    del model, test_data, h1, h2, cm
    tf.keras.backend.clear_session()
    gc.collect()
    print(f"Cleared memory after {name}\n")

df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1"])
df = df.sort_values(by="Accuracy", ascending=False)
df.to_csv("results.csv", index=False)

print("\n===== FINAL RESULT =====", df)

Train model: EfficientNetB0
Found 2513 images belonging to 26 classes.
Found 316 images belonging to 26 classes.
Found 322 images belonging to 26 classes.
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
=====Phase 1 (train head)=====
Epoch 1/15
  5/158 ━━━━━━━━━━━━━━━━━━━━ 10:20 4s/step - accuracy: 0.0160 - loss: 3.3453

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


158/158 ━━━━━━━━━━━━━━━━━━━━ 1104s 7s/step - accuracy: 0.2702 - loss: 2.4891 - val_accuracy: 0.4842 - val_loss: 1.7381
Epoch 2/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 56s 357ms/step - accuracy: 0.4847 - loss: 1.6315 - val_accuracy: 0.5570 - val_loss: 1.4000
Epoch 3/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 57s 362ms/step - accuracy: 0.5591 - loss: 1.3445 - val_accuracy: 0.6266 - val_loss: 1.2150
Epoch 4/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 66s 416ms/step - accuracy: 0.6232 - loss: 1.1658 - val_accuracy: 0.6139 - val_loss: 1.1345
Epoch 5/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 65s 413ms/step - accuracy: 0.6578 - loss: 1.0382 - val_accuracy: 0.6297 - val_loss: 1.1006
Epoch 6/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 60s 378ms/step - accuracy: 0.6769 - loss: 0.9370 - val_accuracy: 0.6203 - val_loss: 1.0559
Epoch 7/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 60s 378ms/step - accuracy: 0.6988 - loss: 0.9207 - val_accuracy: 0.6392 - val_loss: 1.0188
Epoch 8/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 61s 388ms/step - accuracy: 0.7179 - loss: 0.8407 - val_

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


158/158 ━━━━━━━━━━━━━━━━━━━━ 97s 521ms/step - accuracy: 0.1918 - loss: 2.7910 - val_accuracy: 0.3513 - val_loss: 2.1157
Epoch 2/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 56s 356ms/step - accuracy: 0.3709 - loss: 1.9963 - val_accuracy: 0.4494 - val_loss: 1.6915
Epoch 3/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 57s 360ms/step - accuracy: 0.4739 - loss: 1.6276 - val_accuracy: 0.5063 - val_loss: 1.4810
Epoch 4/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 57s 363ms/step - accuracy: 0.5241 - loss: 1.4598 - val_accuracy: 0.5158 - val_loss: 1.4209
Epoch 5/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 57s 364ms/step - accuracy: 0.5527 - loss: 1.3464 - val_accuracy: 0.5190 - val_loss: 1.3943
Epoch 6/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 57s 364ms/step - accuracy: 0.5706 - loss: 1.2369 - val_accuracy: 0.5506 - val_loss: 1.3292
Epoch 7/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 57s 359ms/step - accuracy: 0.5957 - loss: 1.1777 - val_accuracy: 0.5380 - val_loss: 1.2993
Epoch 8/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 56s 356ms/step - accuracy: 0.6216 - loss: 1.1046 - val

...

158/158 ━━━━━━━━━━━━━━━━━━━━ 84s 457ms/step - accuracy: 0.0673 - loss: 3.2556 - val_accuracy: 0.0665 - val_loss: 3.1279
Epoch 2/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 61s 387ms/step - accuracy: 0.0808 - loss: 3.1443 - val_accuracy: 0.0823 - val_loss: 3.0752
Epoch 3/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 61s 385ms/step - accuracy: 0.0915 - loss: 3.0916 - val_accuracy: 0.1044 - val_loss: 3.0240
Epoch 4/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 60s 377ms/step - accuracy: 0.1070 - loss: 3.0592 - val_accuracy: 0.1013 - val_loss: 3.0104
Epoch 5/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 61s 388ms/step - accuracy: 0.1003 - loss: 3.0313 - val_accuracy: 0.1329 - val_loss: 2.9392
Epoch 6/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 60s 378ms/step - accuracy: 0.1182 - loss: 2.9743 - val_accuracy: 0.1013 - val_loss: 2.9236
Epoch 7/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 59s 374ms/step - accuracy: 0.1222 - loss: 2.9557 - val_accuracy: 0.1361 - val_loss: 2.9226
Epoch 8/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 60s 377ms/step - accuracy: 0.1281 - loss: 2.9267 - val

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


158/158 ━━━━━━━━━━━━━━━━━━━━ 125s 618ms/step - accuracy: 0.1277 - loss: 3.0400 - val_accuracy: 0.3259 - val_loss: 2.5032
Epoch 2/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 59s 376ms/step - accuracy: 0.3108 - loss: 2.2922 - val_accuracy: 0.4557 - val_loss: 1.8774
Epoch 3/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 61s 387ms/step - accuracy: 0.4015 - loss: 1.9077 - val_accuracy: 0.5348 - val_loss: 1.5603
Epoch 4/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 60s 380ms/step - accuracy: 0.4723 - loss: 1.6181 - val_accuracy: 0.5791 - val_loss: 1.4336
Epoch 5/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 59s 376ms/step - accuracy: 0.5090 - loss: 1.4922 - val_accuracy: 0.5886 - val_loss: 1.3394
Epoch 6/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 68s 429ms/step - accuracy: 0.5460 - loss: 1.3575 - val_accuracy: 0.6361 - val_loss: 1.2390
Epoch 7/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 68s 428ms/step - accuracy: 0.5702 - loss: 1.2664 - val_accuracy: 0.6171 - val_loss: 1.1992
Epoch 8/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 65s 410ms/step - accuracy: 0.5850 - loss: 1.2334 - va

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


158/158 ━━━━━━━━━━━━━━━━━━━━ 137s 746ms/step - accuracy: 0.1628 - loss: 2.8445 - val_accuracy: 0.3449 - val_loss: 2.2942
Epoch 2/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 93s 585ms/step - accuracy: 0.2949 - loss: 2.2587 - val_accuracy: 0.4367 - val_loss: 1.8989
Epoch 3/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 93s 586ms/step - accuracy: 0.3860 - loss: 1.9364 - val_accuracy: 0.5000 - val_loss: 1.7038
Epoch 4/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 91s 576ms/step - accuracy: 0.4178 - loss: 1.7753 - val_accuracy: 0.4715 - val_loss: 1.6132
Epoch 5/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 91s 577ms/step - accuracy: 0.4361 - loss: 1.6621 - val_accuracy: 0.5158 - val_loss: 1.4820
Epoch 6/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 92s 580ms/step - accuracy: 0.4855 - loss: 1.5694 - val_accuracy: 0.5411 - val_loss: 1.4080
Epoch 7/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 89s 560ms/step - accuracy: 0.5090 - loss: 1.4774 - val_accuracy: 0.5411 - val_loss: 1.4154
Epoch 8/15
158/158 ━━━━━━━━━━━━━━━━━━━━ 93s 588ms/step - accuracy: 0.5261 - loss: 1.4088 - va

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("all_files", 'zip', save_folder)
files.download("all_files.zip")

print('Download file complete')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download file complete
